📝
# FieldCare — Sprint 2 Live notebook

Use this **one Live notebook for all four live sessions in this sprint**. Your Campus lessons have their own Campus notebook. Save a personal Drive copy of each and return to the same copies throughout the sprint.

HelioDesk’s FieldCare service is our shared case. The workshops let you apply the taught patterns, inspect actual observations and defend your decisions with a peer. Each session below names its starting state, task, evidence and debrief.

## Start and preserve your work

1. Select a Python CPU runtime and run setup and imports once per runtime.
2. Upload your current Campus source/evidence checkpoint in Colab’s Files panel. Set `CAMPUS_CHECKPOINT` to its path. A checkpoint is your saved service files, not the notebook itself.
3. Run the workspace cell. It restores into an isolated Live workspace, preserving the Campus copy. With no checkpoint, `USE_DEMO_RECOVERY=True` explicitly selects a labelled public practice state; it does not establish completion of Campus work.
4. Use the table of contents to open the current session. Run its preparation cells before its task. Later sessions build on earlier live sections unless a supplied checkpoint already provides that starting state.
5. Write your own predictions and explanations. Blank decisions stay pending; **Run all does not complete your work**. Public practice never completes the reserved Campus assessment.

Use a new `RUN_NAME` when importing a newer checkpoint; existing work is deliberately preserved. `RESUME_ZIP` reopens an earlier Live export. Never paste keys into cells or outputs. These workshops use demo/transport fixtures by default, not real model output.


## Instructor guides and live decks in Notion

- [LS 5 — API key auth and secrets management](https://app.notion.com/p/3ea5a55972ad815a8aacc8b5e81a2e48?pvs=204)
- [LS 6 — Rate limiting AI endpoints](https://app.notion.com/p/3ea5a55972ad81a4ac37e3e6d49b1933?pvs=204)
- [LS 7 — Secrets safety in practice](https://app.notion.com/p/3ea5a55972ad816f954ff56542286d89?pvs=204)
- [LS 8 — Sprint 2 checkpoint](https://app.notion.com/p/3ea5a55972ad81f89fdfc35996bca7f1?pvs=204)


In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json
import httpx
from module_b.data import load_fixture
from module_b.edits import write_source
from module_b.security import load_callers
from module_b.security_lab import (prepare_security_workspace, runtime_keys, demo_runtime,
    observe_security, review_security, configuration_template_ok)
from module_b.secret_lab import MARKER, marker_present, repository_rehearsal
from module_b.notebook import (table, baseline, register_source, apply_student_edits,
    revision, fresh, export_progress)
from module_b.workspace import restore_workspace

In [ ]:
RUN_NAME = 'sprint-2-live'
CAMPUS_CHECKPOINT = ''  # Current Campus Sprint 2 checkpoint, or your completed Sprint 1 checkpoint
USE_DEMO_RECOVERY = True
RESUME_ZIP = ''
RESUME_STAGE = 'practice'  # auth or live-limit, as printed at export
work = REPO/'work'/RUN_NAME
if RESUME_ZIP:
    assert RESUME_STAGE in ('auth','live-limit')
    if not (work/RESUME_STAGE).exists():
        restore_workspace(RESUME_ZIP,work/RESUME_STAGE)
if not CAMPUS_CHECKPOINT and not USE_DEMO_RECOVERY:
    raise ValueError('Set your Sprint 1 checkpoint path, or explicitly choose demo recovery.')
auth, auth_origin = prepare_security_workspace(work/'auth',checkpoint=CAMPUS_CHECKPOINT or None,repo_root=REPO)
accepted = load_fixture(auth,'valid-request')
CALLER_ENV = {'dispatch':'FIELDCARE_DISPATCH_KEY','partner':'FIELDCARE_PARTNER_KEY'}
evidence = {}
print('Starting source:',auth_origin['provenance'])
print('Authentication stage:',auth)

## Live workshops · access, allowance and secret paths

Run setup, imports and the workspace cell above once, then use the relevant workshop section. FieldCare is a supplied synthetic equipment-support service with versioned POST operations. Your incoming service remains in `auth`; the limiter experiment uses a separate copy. No prior business-story knowledge is required.

| Session | Work | Individual evidence |
|---|---|---|
| LS05 | Apply/audit caller authentication | All-route accepted/rejected matrix and safe startup restoration |
| LS06 | Apply the supplied two-request limiter | Ordered burst, caller isolation and timed renewal |
| LS07 | Apply three bounded secret protections | Saved log, outbound payload and Git observations with a fresh marker |
| LS08 | Defend the controlled service | Public evidence index, personal challenge response and repeated check |

Use one personal notebook and its single final export. Live records are formative: they never complete the independent Campus assessment or award a grade. Demo HTTP and separate controlled-clock probes establish different observations; no external provider call is made.


In [ ]:
from module_b.live_sessions import save_record,read_records
from module_b.secret_workshop import prepare_secret_workshop,inspect_secret_workshop,git_fixture
# Same two public practice caller labels; values are generated in memory only.
LIVE_CALLERS = {'dispatch':'FIELDCARE_DISPATCH_KEY','partner':'FIELDCARE_PARTNER_KEY'}
live_request = {'question':'Which HX filter and airflow checks should I try?', 'equipment_id':'EQ-FC-1002'}
live_env = runtime_keys(LIVE_CALLERS)
print('Retained incoming service:',auth)
print('Caller configuration names:',LIVE_CALLERS)
print('Saved live sessions:',list(read_records(auth)))


### LS05 · Caller and provider credentials have different destinations

The caller presents `X-API-Key` to this service. A provider credential, when live mode is used, belongs to the service's model adapter. Neither belongs in task text or saved evidence. **Authentication** maps a recognized runtime credential to a caller label; it does not prove answer quality or a person's identity.

**Apply:** inspect the supplied configuration and attachment below, then set `APPLY_LS05=True` to use them on your incoming service. The guard must be attached after all route registrations. An existing attachment is retained; inspect its order instead of adding a second guard. Existing configuration is preserved. If you deliberately need to change it, edit `app/security_settings.py` and rerun the observations.

**Expected:** every static POST version is inventoried; public health/docs stay available. Missing/wrong callers receive `401` before adapter work. Values enter the child process in memory, not through a committed `.env` or a printed header. The helper does not automatically load a `.env` file.


In [ ]:
APPLY_LS05 = False  # Read and predict before applying the supplied pattern.
live_auth_source = (REPO/'examples/patterns/security_settings.py').read_text()
live_attachment = '\nfrom module_b.security import SecurityMiddleware, protected_post_paths\nfrom app.security_settings import CALLER_ENV, POLICY\napp.add_middleware(SecurityMiddleware, caller_env=CALLER_ENV,\n                   protected_paths=protected_post_paths(app), policy=POLICY)\n'
print(live_auth_source)
print(live_attachment)
if APPLY_LS05:
    if not (auth/'app/security_settings.py').exists():
        write_source(auth,'app/security_settings.py',live_auth_source)
    if 'app.add_middleware(SecurityMiddleware' not in (auth/'app/main.py').read_text():
        register_source(auth,live_attachment)
    print((auth/'app/main.py').read_text())
ls05_inventory = observe_security(auth,caller_env=LIVE_CALLERS,sequence=[])
ls05_paths = ls05_inventory.get('routes',[])
print('Guard startup:',ls05_inventory.get('startup'))
print('Protected inventory:',ls05_paths)


### LS05 · Two kinds of execution evidence

Predict allowed/missing/wrong results for **every** registered POST path, including old or hidden routes. Run the cell after the attachment is ready. Actual local HTTP verifies the caller-visible contract. The separate new in-process requests count adapter entries; they are not instrumentation attached to the preceding HTTP calls.

The safe missing-configuration probe removes one required variable at a time. Expect startup refusal. Restoring the complete environment and starting an actual service again should allow a supported call. At the checkpoint, explain why a `401` alone cannot prove the adapter was skipped.


In [ ]:
ls05_observations = {}
if ls05_inventory.get('startup')=='ready' and (ls05_inventory.get('configuration') or {}).get('caller_env')==LIVE_CALLERS:
    ls05_http=[]
    with demo_runtime(auth,live_env) as service:
        for path in ls05_paths:
            for caller in ('dispatch','partner','missing','wrong'):
                headers={} if caller=='missing' else {'X-API-Key':live_env[LIVE_CALLERS[caller]] if caller in LIVE_CALLERS else 'invalid-local-test'}
                response=httpx.post(service.base_url+path,json=live_request,headers=headers)
                ls05_http.append({'caller':caller,'path':path,'status':response.status_code})
        ls05_public={p:httpx.get(service.base_url+p).status_code for p in ('/health','/docs','/openapi.json')}
    ls05_probe=observe_security(auth,caller_env=LIVE_CALLERS,sequence=[
        {'caller':caller,'path':path,'payload':live_request} for path in ls05_paths for caller in ('dispatch','partner','missing','wrong')])
    ls05_missing=[{'name':name,'startup':observe_security(auth,caller_env=LIVE_CALLERS,sequence=[],missing_env=name).get('startup')} for name in LIVE_CALLERS.values()]
    with demo_runtime(auth,live_env) as service:
        restored=httpx.post(service.base_url+'/v1/diagnose',json=live_request,headers={'X-API-Key':live_env[LIVE_CALLERS['dispatch']]})
    ls05_observations={'http':ls05_http,'public':ls05_public,'separate_probe':ls05_probe,'missing_configuration':ls05_missing,'restored_http':restored.status_code}
    table(ls05_http);table(ls05_probe.get('rows',[]),['caller','path','status','adapter_calls']);table(ls05_missing)
else:
    print('Pending: apply/repair the supplied auth configuration and rerun its inventory.')
# During the 14-minute task, let a peer choose a route/condition from your public card.
ls05_reasoning={'two_credentials':'','coverage_and_guard_order':'','runtime_restoration':'','peer_challenge_and_response':''}
print(save_record(auth,5,observations=ls05_observations,reasoning=ls05_reasoning,origin='learner_service')['review_status'])


### LS06 · One caller owns one allowance across versions

**Apply:** after auth is working, create the separate `live_limit` from it and set `APPLY_LS06=True`. The supplied policy permits **2 admitted requests per caller per 60 seconds**, across every protected route. Its fixed window begins with that caller's first admission. This workshop keeps the two-request worked policy; the independent Campus exercise's three-request experiment remains yours to design.

The order is authentication → admission → input validation → handler → possible adapter. Missing/wrong keys return `401` without allowance use. An admitted invalid body consumes allowance and returns `422` without adapter work. Exhaustion returns `429` and `Retry-After`; it does not invoke the adapter. This is one process's memory, not a deployment-wide quota or spending limit.


In [ ]:
APPLY_LS06 = False
live_limit = None
ls06_inventory = {}
if ls05_inventory.get('startup')=='ready' and ls05_paths:
    live_limit,_=prepare_security_workspace(work/'live-limit',checkpoint=auth,repo_root=REPO)
    ls06_source = "from module_b.security import LimitPolicy\nCALLER_ENV = {'dispatch':'FIELDCARE_DISPATCH_KEY','partner':'FIELDCARE_PARTNER_KEY'}\nPOLICY = LimitPolicy(allowance=2, window_seconds=60)\n"
    print(ls06_source)
    if APPLY_LS06:
        source_path=live_limit/'app/security_settings.py'
        write_source(live_limit,'app/security_settings.py',ls06_source,expected=source_path.read_text())
    ls06_inventory=observe_security(live_limit,caller_env=LIVE_CALLERS,sequence=[])
print('Observed policy:',ls06_inventory.get('configuration'))


In [ ]:
ls06_demo={}
if live_limit is not None and (ls06_inventory.get('configuration') or {}).get('allowance')==2:
    ls06_http=[]
    with demo_runtime(live_limit,live_env) as service:
        for caller in ('dispatch','dispatch','dispatch','partner'):
            response=httpx.post(service.base_url+'/v1/diagnose',json=live_request,headers={'X-API-Key':live_env[LIVE_CALLERS[caller]]})
            ls06_http.append({'caller':caller,'status':response.status_code,'retry_after':response.headers.get('retry-after')})
    # ONE fresh controlled-clock worker; 'at' is simulated elapsed time, not a sleep.
    ls06_other=next(p for p in ls06_inventory['routes'] if p!='/v1/diagnose')
    ls06_guided=[
        {'caller':'dispatch','path':'/v1/diagnose','at':0},
        {'caller':'dispatch','path':ls06_other,'at':0},
        {'caller':'dispatch','path':'/v1/diagnose','at':0},
        {'caller':'partner','path':'/v1/diagnose','at':10},
        {'caller':'dispatch','path':'/v1/diagnose','at':60},
        {'caller':'partner','path':ls06_other,'at':60},
        {'caller':'partner','path':'/v1/diagnose','at':60},
        {'caller':'partner','path':'/v1/diagnose','at':70}]
    ls06_timeline=observe_security(live_limit,caller_env=LIVE_CALLERS,sequence=ls06_guided)
    ls06_demo={'http':ls06_http,'controlled_clock':ls06_timeline}
    table(ls06_http);table(ls06_timeline['rows'],['at','caller','path','status','adapter_calls','retry_after'])
else:
    print('Pending: apply the two-request workshop policy.')


### LS06 · A changed order, 14 minutes

Predict each row below before setting `RUN_LS06_TASK=True`. Partner starts first; one admitted body is deliberately invalid; dispatch starts later. Select the rows that establish route pooling, independent callers and renewal without restarting. A second auth rejection remains in the sequence so you can distinguish access from usage control.

**Keep:** your predictions, the observed ordered table, an explanation of a mismatch or justified match, and the single-process limitation. Changing `at` advances the injected test clock; it does not measure real waiting or production throughput. Do not combine results from separate probe calls into one timeline.


In [ ]:
RUN_LS06_TASK = False
ls06_predictions=[]
ls06_task={}
if live_limit is not None and ls06_demo:
    ls06_changed=[
        {'caller':'missing','path':'/v1/diagnose','at':0},
        {'caller':'partner','path':'/v1/diagnose','at':0,'payload':{}},
        {'caller':'partner','path':ls06_other,'at':1},
        {'caller':'partner','path':'/v1/diagnose','at':2},
        {'caller':'dispatch','path':'/v1/diagnose','at':10},
        {'caller':'wrong','path':'/v1/diagnose','at':10},
        {'caller':'dispatch','path':ls06_other,'at':11},
        {'caller':'partner','path':'/v1/diagnose','at':60},
        {'caller':'dispatch','path':'/v1/diagnose','at':60},
        {'caller':'dispatch','path':'/v1/diagnose','at':70}]
    if RUN_LS06_TASK:
        ls06_task=observe_security(live_limit,caller_env=LIVE_CALLERS,sequence=ls06_changed)
        table(ls06_task.get('rows',[]),['at','caller','path','status','adapter_calls','retry_after'])
ls06_reasoning={'counting':'','isolation_and_pooling':'','renewal':'','predicted_vs_observed':'','scope_limit':''}
print(save_record(auth,6,observations={'task':ls06_task,'predictions':ls06_predictions} if ls06_task else {},reasoning=ls06_reasoning,origin='learner_service',observed_project=live_limit or auth)['review_status'])


### LS07 · Three deliberately unsafe miniature fixtures

A **sink** is where a value is written or sent to an audience: saved log output, an outbound prompt, or repository history. The marker below cannot authenticate anywhere. The helper creates only a dedicated disposable repository; it does not inspect your personal repositories or use a provider.

First run the unmodified fixture and inspect the saved files. Each prevention has its own apply cell. Read the supplied source, predict the changed check, then set its flag to `True` and run it. Every learner performs all three; do not divide the paths between partners.


In [ ]:
secret_fixture=prepare_secret_workshop(work/'live-secret-guided',marker='SYNTHETIC_GUIDED_NOT_A_CREDENTIAL',repo_root=REPO)
ls07_before=inspect_secret_workshop(secret_fixture)
table([{'check':k,'observed':v} for k,v in ls07_before['checks'].items()])
print('Synthetic saved log:',(secret_fixture/'observed.log').read_text())
print('Synthetic outbound payload:',(secret_fixture/'outbound-payload.json').read_text())


In [ ]:
# Cycle 1: select safe fields BEFORE writing ordinary and exception output.
APPLY_LS07_LOG = False
log_prevention=(REPO/'examples/patterns/secret_workshop/log_safe.py').read_text()
print(log_prevention)
if APPLY_LS07_LOG:
    (secret_fixture/'log_policy.py').write_text(log_prevention)
ls07_log=inspect_secret_workshop(secret_fixture)
print('Saved ordinary/error log:',(secret_fixture/'observed.log').read_text())
print({k:v for k,v in ls07_log['checks'].items() if 'log' in k or 'diagnostic' in k})


In [ ]:
# Cycle 2: construct only the task payload; do not include runtime configuration.
APPLY_LS07_PROMPT = False
prompt_prevention=(REPO/'examples/patterns/secret_workshop/prompt_safe.py').read_text()
print(prompt_prevention)
if APPLY_LS07_PROMPT:
    (secret_fixture/'prompt_policy.py').write_text(prompt_prevention)
ls07_prompt=inspect_secret_workshop(secret_fixture)
print('Local payload; never sent:',(secret_fixture/'outbound-payload.json').read_text())
print({k:v for k,v in ls07_prompt['checks'].items() if 'prompt' in k or 'context' in k})


In [ ]:
# Cycle 3: future exclusion is separate from historical exposure.
APPLY_LS07_REPOSITORY = False
if APPLY_LS07_REPOSITORY:
    (secret_fixture/'.gitignore').write_text('.env\nlocal.key\n')
    print('Already tracked after ignore:',bool(git_fixture(secret_fixture,'ls-files','.env')))
    # Idempotent on rerun; untrack only the harmless synthetic fixture file.
    if git_fixture(secret_fixture,'ls-files','.env'):
        git_fixture(secret_fixture,'rm','--cached','.env')
ls07_git=inspect_secret_workshop(secret_fixture)
print({k:v for k,v in ls07_git['checks'].items() if 'ignored' in k or 'untracked' in k or 'historical' in k})
print('This does not erase history or revoke exposed access.')


### LS07 · Fresh-marker verification, 15 minutes

Set `RUN_LS07_TASK=True` after the three guided changes. A new disposable fixture uses a **different** nonfunctional marker. Carry your own prevention source and exclusion rule into it, then inspect the actual saved artifacts. Do not hardcode removal of the first marker. This task changes the data and asks you to choose the decisive evidence, rather than introducing another security mechanism.

Keep a separate **source → sink → prevention → observed check** record for logs, prompts and repository inclusion. Include both ordinary/error logging, useful task/diagnostic content, and the historical marker that remains. Map each prevention to the corresponding boundary in your own service; fixture success alone cannot establish that a different project path is protected. If a real credential were exposed, the response would require invalidating/replacing access and following the organization's remediation process; no real rotation or history rewriting is performed here.


In [ ]:
RUN_LS07_TASK = False
ls07_fresh={}
if RUN_LS07_TASK:
    fresh_fixture=prepare_secret_workshop(work/'live-secret-fresh',marker='SYNTHETIC_FRESH_NOT_A_CREDENTIAL',repo_root=REPO)
    for name in ('log_policy.py','prompt_policy.py','.gitignore'):
        if (secret_fixture/name).is_file():
            (fresh_fixture/name).write_text((secret_fixture/name).read_text())
    if (fresh_fixture/'.gitignore').is_file() and git_fixture(fresh_fixture,'ls-files','.env'):
        git_fixture(fresh_fixture,'rm','--cached','.env')
    ls07_fresh=inspect_secret_workshop(fresh_fixture)
    table([{'check':k,'observed':v} for k,v in ls07_fresh['checks'].items()])
ls07_reasoning={'log_source_sink_prevention_check':'','prompt_source_sink_prevention_check':'','repository_source_sink_prevention_check':'','prior_exposure_response':'','own_service_mapping':''}
# Preserve the applied source with the evidence; the service itself was not edited by this lab.
ls07_sources={name:(secret_fixture/name).read_text() for name in ('log_policy.py','prompt_policy.py','.gitignore') if (secret_fixture/name).is_file()}
print(save_record(auth,7,observations={'fresh_marker':ls07_fresh,'applied_fixture_source':ls07_sources} if ls07_fresh else {},reasoning=ls07_reasoning,origin='public_practice')['review_status'])


### LS08 · An individual control claim

Bring public familiar-project observations, not the reserved assessment brief, payloads or implementation. Use three eight-minute owner rounds in triads: four minutes demonstrate, two challenge, one feedback, one record/rotate. Each owner must show accepted/missing/wrong auth, safe configuration restoration, two callers, exhausted allowance, timed renewal and zero adapter work on rejections. Review the relevant prior secret-boundary evidence on your actual project.

After the rounds, spend **12 minutes** on one narrow repair or missing check. Rerun the affected public request and a neighbouring allowed path. Record whether the evidence is your service execution, public recovery or recorded replay. An unexecuted repair stays pending; this review creates no grade.

Choose `ls08_project` deliberately. A workshop copy is suitable for formative reasoning, but label it; it cannot substitute for a missing personal implementation. Use the same caller/policy as that project. The optional probe below reuses only the public workshop sequence when that exact two-request configuration is present.


In [ ]:
ls08_project=live_limit or auth
RUN_LS08_CHECK = False
ls08_observations={}
if RUN_LS08_CHECK and ls06_task:
    ls08_observations=observe_security(ls08_project,caller_env=LIVE_CALLERS,sequence=ls06_changed)
    table(ls08_observations.get('rows',[]),['at','caller','path','status','adapter_calls','retry_after'])
ls08_reasoning={'public_evidence_index':'','control_claim':'','peer_challenge':'','owner_response':'','repair_or_no_change':'','repeated_observation_or_pending':'','secret_prevention_on_own_service':''}
print(save_record(auth,8,observations=ls08_observations,reasoning=ls08_reasoning,origin='learner_service',observed_project=ls08_project)['review_status'])
print('Use the final Save section once; it retains these records with your source checkpoint.')


## Save your Live evidence and carry improvements back

Run this export after your session and again after later changes. It saves the Live workspace and the four public workshop records; it does not mark your Campus assessment complete. Download the printed ZIP from Colab’s Files panel and save this Live notebook separately.

**Apply a chosen improvement to Campus:** open your Campus copy, compare the changed source file with your own file, and apply only the change you can explain using the Campus editing cell or workspace editor. Rerun the affected route/control checks and the preservation checks. Then export a new Campus checkpoint. Do not replace the entire Campus project with a worked Live example. Your next sprint starts from the latest verified **Campus** checkpoint.

**Expected result:** the ZIP contains your Live source and safe observations, all session records remain available on resume, and your Campus workspace has not been changed by this notebook.


In [ ]:
from module_b.live_sessions import read_records
final_project = globals().get('live_limit') or auth
final_stage = 'live-limit' if final_project.name == 'live-limit' else 'auth' if final_project.name == 'auth' else 'live-project'
live_records = {**read_records(auth), **read_records(final_project)}
(final_project/'fixtures/live-session-records.json').write_text(json.dumps(live_records, indent=2)+'\n')
readiness = {'structural_ready': False, 'written_evidence_present': False,
    'scope': 'live_formative_only', 'campus_assessment': 'not_evaluated',
    'session_review_status': {k:v['review_status'] for k,v in live_records.items()}}
SPRINT2_EXPORT = export_progress(final_project, REPO/'artifacts'/f'sprint-2-live', sprint=2,
    evidence={'live_sessions': live_records}, notes={'purpose':'Live workshop evidence; apply chosen changes back to Campus explicitly.'}, readiness=readiness)
print('Resume stage:', final_stage)
print('Live review status:', readiness['session_review_status'])


## Watch out for

| Observation | Check and recovery |
|---|---|
| Your new checkpoint was not imported | Existing Live work is preserved. Choose a new `RUN_NAME`, then rerun the workspace cell. |
| A later session has no prepared service | Run the named earlier guided cells or import the correct checkpoint; do not infer completion from a blank task. |
| A task still says pending | Fill your decisions and reasoning, execute its checks, then export again. |
| Runtime ended before saving | Restore your downloaded Live ZIP in a fresh runtime; saving only the notebook does not preserve workspace files. |

## Quick reference

| Action | Location |
|---|---|
| Shared code and dependency installation | `setup` |
| Import Campus or resume Live work | `workspaces` |
| Session activity and evidence | Named LS section |
| Save source and public observations | `save-checkpoint` |

## Summary

You have a separate place to rehearse, investigate and discuss FieldCare service behavior. Keep the observations tied to the source you ran, carry justified changes back into your Campus project, and preserve both your notebook and source/evidence checkpoint.
